# Interesting Plots

A collection of visualizations demonstrating satkit's atmospheric density models, gravity field, and coordinate frame transformations.

## Air Density as a Function of the Solar Cycle

Atmospheric density at orbital altitudes varies by more than an order of magnitude over the ~11-year solar cycle. During solar maximum, increased UV and EUV radiation heats the upper atmosphere, causing it to expand and increasing drag on satellites. This effect dominates orbit lifetime predictions for low-Earth orbit spacecraft.

The plot below uses the NRLMSISE-00 density model at 400 km and 500 km altitude, showing density variations from 1995 to 2022 (spanning roughly two solar cycles).

In [ ]:
import satkit as sk
import matplotlib.pyplot as plt
import scienceplots  # noqa: F401
plt.style.use(["science", "no-latex", "../satkit.mplstyle"])
%config InlineBackend.figure_formats = ['svg']
import numpy as np
import math as m

start = sk.time(1995, 1, 1)
end = sk.time(2022, 12, 31)
duration = end - start
timearray = np.array([start + sk.duration(days=x) for x in np.linspace(0, duration.days, 4000)])

altitude = 400e3
rho_400, _temperature = zip(*np.array([sk.density.nrlmsise(altitude, 0, 0, x) for x in timearray]))

altitude = 500e3
rho_500, emperature = zip(*np.array([sk.density.nrlmsise(altitude, 0, 0, x) for x in timearray]))



fig, ax = plt.subplots(figsize=(8, 5))
dates = [t.as_datetime() for t in timearray]
ax.semilogy(dates, rho_400, "k-", linewidth=1, label="Altitude = 400km")
ax.semilogy(dates, rho_500, "k--", linewidth=1, label="Altitude = 500km")
ax.set_xlabel("Year")
ax.set_ylabel("Density [kg/m$^3$]")
ax.set_title("Air Density Changes with Solar Cycle")
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## Forces Acting on a Satellite vs. Altitude

This plot (inspired by Montenbruck & Gill, *Satellite Orbits*) shows the magnitude of various perturbation accelerations as a function of distance from the Earth's center. Key observations:

- **Gravity** dominates at all altitudes
- **J2** (Earth's oblateness) is the largest perturbation, ~1000x stronger than higher-order terms
- **Atmospheric drag** is significant below ~800 km and varies with the solar cycle (max vs. min curves)
- **Solar radiation pressure** is roughly constant with altitude
- **Third-body** effects (Sun, Moon) grow with altitude as the satellite moves further from the Earth

In [ ]:
## Force components acting on satellite vs altitude
import numpy as np
import satkit as sk
import math as m
import matplotlib.pyplot as plt
import scienceplots  # noqa: F401
plt.style.use(["science", "no-latex", "../satkit.mplstyle"])
%config InlineBackend.figure_formats = ['svg']


N = 1000
range_arr = np.logspace(m.log10(6378.2e3 + 100e3), m.log10(50e6), N)
grav1v = np.array([sk.gravity(np.array([a, 0, 0]), order=1) for a in range_arr])
grav1 = np.linalg.norm(grav1v, axis=1)
grav2v = np.array([sk.gravity(np.array([a, 0, 0]), order=2) for a in range_arr])
grav2 = np.linalg.norm(grav2v-grav1v, axis=1)

grav6v = np.array([sk.gravity(np.array([a, 0, 0]), order=6) for a in range_arr])
grav5v = np.array([sk.gravity(np.array([a, 0, 0]), order=5) for a in range_arr])
grav6 = np.linalg.norm(grav6v-grav5v, axis=1)

aoverm = 0.01
Cd = 2.2
Cr = 1.0

# Drag force at maximum & minimum density
didx = np.argwhere(range_arr - sk.consts.earth_radius < 800e3).flatten()
tm_max = sk.time(2001, 12, 1)
tm_min = sk.time(1996, 12, 1)
rho_max = np.array([sk.density.nrlmsise(a-sk.consts.earth_radius, 0, 0, tm_max)[0] for a in range_arr[didx]])
rho_min = np.array([sk.density.nrlmsise(a-sk.consts.earth_radius, 0, 0, tm_min)[0] for a in range_arr[didx]])
varr = np.sqrt(sk.consts.mu_earth/(range_arr + sk.consts.earth_radius))
drag_max = 0.5 * rho_max * varr[didx]**2 * Cd * aoverm
drag_min = 0.5 * rho_min * varr[didx]**2 * Cd * aoverm

moon_range = np.linalg.norm(sk.jplephem.geocentric_pos(sk.solarsystem.Moon, sk.time(2023, 1, 1)))
moon = sk.consts.mu_moon*((moon_range-range_arr)**(-2) - moon_range**(-2))
sun = sk.consts.mu_sun*((sk.consts.au-range_arr)**(-2) - sk.consts.au**(-2))

a_radiation = 4.56e-6 * 0.5 * Cr * aoverm * np.ones(range_arr.shape)


fig, ax = plt.subplots(figsize=(8, 8))
def add_line(ax, x, y, text, frac=0.5, dx=-20, dy=-20):
    ax.loglog(x, y, "k-", linewidth=1.5)
    idx = int(len(x)*frac)
    ax.annotate(text, xy=(x[idx], y[idx]), fontsize=9,
                xytext=(dx, dy), textcoords="offset points",
                arrowprops=dict(arrowstyle="->", color="gray"))

add_line(ax, range_arr/1e3, grav1/1e3, "Gravity")
add_line(ax, range_arr/1e3, grav2/1e3, "J2", 0.2, 0, -15)
add_line(ax, range_arr/1e3, grav6/1e3, "J6", 0.8, 0, -15)
add_line(ax, range_arr[didx]/1e3, drag_max/1e3, "Drag Max", 0.7, 30, 0)
add_line(ax, range_arr[didx]/1e3, drag_min/1e3, "Drag Min", 0.8, 10, 30)
add_line(ax, range_arr/1e3, moon/1e3, "Moon", 0.8, -10, -15)
add_line(ax, range_arr/1e3, sun/1e3, "Sun", 0.7, -10, 15)
add_line(ax, range_arr/1e3, a_radiation/1e3, "Radiation\nPressure", 0.3, -10, 15)
ax.set_xlabel("Distance from Earth Origin [km]")
ax.set_ylabel("Acceleration [km/s$^2$]")
ax.set_title("Satellite Forces vs Altitude")
ax.set_xlim(6378.1, 50e3)
plt.tight_layout()
plt.show()

## Difference in Angle Between IERS 2010 Reduction and Approximate Calculation

`satkit` provides both the full IERS 2010 precession/nutation model (`qgcrf2itrf`) and a faster approximate version (`qgcrf2itrf_approx`). This plot shows the angular difference between the two over several decades. The error remains below a few arcseconds, making the approximate model suitable for many applications where speed matters more than sub-arcsecond accuracy.

In [ ]:
import matplotlib.pyplot as plt
import scienceplots  # noqa: F401
plt.style.use(["science", "no-latex", "../satkit.mplstyle"])
%config InlineBackend.figure_formats = ['svg']
import numpy as np
import satkit as sk
import math as m

start = sk.time(2023, 1, 1)
end = sk.time(1970, 1, 1)
duration = end - start
timearray = np.array([start + sk.duration(days=x) for x in np.linspace(0, duration.days, 4000)])
qexact = sk.frametransform.qgcrf2itrf(timearray)
qapprox = sk.frametransform.qgcrf2itrf_approx(timearray)
qdiff = np.array([q1 * q2.conj for q1, q2 in zip(qexact, qapprox)]) # type: ignore
theta = np.array([min(q.angle, 2*m.pi - q.angle) for q in qdiff])
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot([t.as_datetime() for t in timearray], theta*180.0/m.pi*3600, "k-", linewidth=1)
ax.set_xlabel("Year")
ax.set_ylabel("Error [arcsec]")
ax.set_title("Angle Error of Approximate ITRF to GCRF Rotation")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## Polar Motion

The Earth's rotational axis wobbles slightly relative to its crust, a phenomenon known as polar motion. The plot below shows the x and y components of polar motion (in arcseconds) over several decades. The dominant signal is the ~14-month Chandler wobble superimposed on an annual cycle, with a slow secular drift.

In [ ]:
import matplotlib.pyplot as plt
import scienceplots  # noqa: F401
plt.style.use(["science", "no-latex", "../satkit.mplstyle"])
%config InlineBackend.figure_formats = ['svg']
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.dates as mdates
import numpy as np
import satkit as sk
import math as m

start = sk.time(2023, 1, 1)
end = sk.time(1970, 1, 1)
duration = end - start
timearray = np.array([start + sk.duration(days=x) for x in np.linspace(0, duration.days, 4000)])

dut1, xp, yp, lod, dX, dY = zip(*[sk.frametransform.earth_orientation_params(t) for t in timearray])
fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(111, projection="3d")
dates_num = mdates.date2num([t.as_datetime() for t in timearray])
ax.plot(list(xp), list(yp), dates_num, linewidth=0.5)
ax.set_xlabel("X [arcsec]")
ax.set_ylabel("Y [arcsec]")
ax.set_zlabel("Year")
ax.set_title("Polar Motion")
plt.tight_layout()
plt.show()

## Precession / Nutation

Precession and nutation describe the long-term and short-term wobble of the Earth's rotational axis in inertial space. This is captured by the rotation between the GCRS (Geocentric Celestial Reference System) and CIRS (Celestial Intermediate Reference System) frames. The plot shows the nutation components (with the linear precession trend removed from x) over a 20-year period; the dominant ~18.6-year and semi-annual periods are clearly visible.

In [ ]:
import satkit as sk
import numpy as np
import math as m
import matplotlib.pyplot as plt
import scienceplots  # noqa: F401
plt.style.use(["science", "no-latex", "../satkit.mplstyle"])
%config InlineBackend.figure_formats = ['svg']
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.dates as mdates

# Precession / Nutation is the difference between the IERS 2010 GCRS frame and the CIRS frame
start = sk.time(2000, 1, 1)
end = sk.time(2020, 1, 1)
duration = end - start
N = 1000
timearray = np.array([start + sk.duration(days=x) for x in np.linspace(0, duration.days, N)])
qarray = np.array([sk.frametransform.qcirs2gcrf(t) for t in timearray])
theta = np.fromiter((q.angle for q in qarray), float)
rots = np.array([q * np.array([0.0, 0.0, 1.0]) for q in qarray])
pline = np.linspace(0,1,N)
pv = np.polyfit(pline, rots[:,0], 1)
rx0 =rots[:,0] - np.polyval(pv, pline)

rad2asec = 180.0/m.pi * 3600
fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(111, projection="3d")
dates_num = mdates.date2num([t.as_datetime() for t in timearray])
ax.plot(rx0*rad2asec, rots[:,1]*rad2asec, dates_num, "k-", linewidth=0.5)
ax.set_xlabel("X [arcsec]")
ax.set_ylabel("Y [arcsec]")
ax.set_zlabel("Year")
ax.set_title("Precession / Nutation")
plt.tight_layout()
plt.show()